In [0]:
%run "../lib/00_common"

In [0]:
%pip install polars

In [0]:
import polars as pl

SCHEMAS = {
    "olist_customers_dataset.csv": {
        "customer_zip_code_prefix": pl.String,
    },
    "olist_geolocation_dataset.csv": {
        "geolocation_zip_code_prefix": pl.String,
    },
    "olist_sellers_dataset.csv": {
        "seller_zip_code_prefix": pl.String,
    },
    "olist_products_dataset.csv": {
        "product_weight_g":  pl.Float64,
        "product_length_cm": pl.Float64,
        "product_height_cm": pl.Float64,
        "product_width_cm":  pl.Float64,
    },
    "olist_orders_dataset.csv": {
        "order_purchase_timestamp":      pl.Datetime,
        "order_approved_at":             pl.Datetime,
        "order_delivered_carrier_date":  pl.Datetime,
        "order_delivered_customer_date": pl.Datetime,
        "order_estimated_delivery_date": pl.Datetime,
    },
    "olist_order_items_dataset.csv": {
        "shipping_limit_date": pl.Datetime,
    },
}

In [0]:
# Recorre todas las subcarpetas de RAW y mapea automáticamente cada CSV encontrado a su destino Parquet en Bronze,
# manteniendo la misma estructura de carpetas.

from pathlib import Path

def build_file_map(raw_root: str, bronze_root: str) -> dict:
    file_map = {}
    
    for csv_file in Path(raw_root).rglob("*.csv"):
        # Ruta relativa respecto a RAW_ROOT
        # Ej: "orders/olist_orders_dataset.csv"
        relative = csv_file.relative_to(raw_root)
        
        # Destino en Bronze con misma estructura, cambiando extensión
        dest = Path(bronze_root) / relative.with_suffix(".parquet")
        
        file_map[str(csv_file)] = str(dest)
    
    return file_map

file_map = build_file_map(RAW_ROOT, BRONZE_ROOT)

print(f"Archivos encontrados: {len(file_map)}\n")

for src, dest in file_map.items():
    print(f"  {Path(src).name:55s} → {Path(dest).name}")

In [0]:

# Función de ingesta RAW → Bronze
# Función de ingesta RAW → Bronze con esquemas corregidos
import os
import polars as pl
import pandas as pd
from pathlib import Path

def ingest_to_bronze(src_csv: str, dest_parquet: str) -> None:
    now      = pd.Timestamp.utcnow()
    run_date = now.date().isoformat()
    filename = Path(src_csv).name

    # 1. Leer CSV con Polars aplicando esquema correcto si existe
    schema_overrides = SCHEMAS.get(filename, {})
    df = pl.read_csv(src_csv, schema_overrides=schema_overrides)

    # 2. Agregar columnas de auditoría
    df = df.with_columns([
        pl.lit(str(now)).alias("_ingestion_ts"),
        pl.lit(run_date).alias("_run_date"),
        pl.lit(filename).alias("_source_file"),
    ])

    # 3. Escribir como Parquet en Bronze
    os.makedirs(os.path.dirname(dest_parquet), exist_ok=True)
    df.write_parquet(dest_parquet)

In [0]:
# Ejecutar ingesta

print("Iniciando ingesta RAW → Bronze...\n")

errores = []

for src, dest in file_map.items():
    try:
        ingest_to_bronze(src, dest)
        print(f"✓ {Path(dest).name}")
    except Exception as e:
        errores.append((src, str(e)))
        print(f"✗ ERROR en {Path(src).name}: {e}")

print(f"\n{' Ingesta completada sin errores.' if not errores else f' Completado con {len(errores)} error(es).'}")

In [0]:
# Verificación
import pyarrow.parquet as pq
print("Verificando archivos en Bronze...\n")

for dest in file_map.values():
    if os.path.exists(dest):
        table = pq.read_table(dest)
        print(f"✓ {Path(dest).name:50s} | filas: {len(table):>7,} | cols: {len(table.schema)}")
    else:
        print(f"✗ No encontrado: {dest}")

In [0]:

# Ver esquemas actuales de los Parquet en Bronze

import pyarrow.parquet as pq
from pathlib import Path

print("Esquemas actuales en Bronze:\n")

for dest in file_map.values():
    if os.path.exists(dest):
        schema = pq.read_schema(dest)
        print(f"{'='*60}")
        print(f"📄 {Path(dest).name}")
        print(f"{'='*60}")
        for field in schema:
            print(f"  {field.name:45s} {str(field.type)}")
        print()